In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# =========================
# Config
# =========================

catalog_name = "jarvis_training"

bronze_schema_name = "bronze"
silver_schema_name = "silver"

source_system = "sqlserver"

lst = ["dbo.cards_data", "dbo.transactions_data", "dbo.users_data"]

# Create silver schema if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema_name}")

# =========================
# Helper Function: Clean Column Names
# =========================

def clean_column_names(df):
    """
    Convert column names into a standard lowercase snake_case format.
    Example:
        Card ID -> card_id
        User.Name -> user_name
    """
    for col_name in df.columns:
        new_col_name = (
            col_name
            .strip()
            .lower()
            .replace(" ", "_")
            .replace(".", "_")
            .replace("-", "_")
            .replace("(", "")
            .replace(")", "")
        )
        df = df.withColumnRenamed(col_name, new_col_name)
    return df


# =========================
# Silver Processing Loop
# =========================

for source_table in lst:

    # Example:
    # source_table = "dbo.cards_data"
    # table_suffix = "dbo_cards_data"

    table_suffix = source_table.replace(".", "_")

    bronze_table = f"{catalog_name}.{bronze_schema_name}.raw_{source_system}_{table_suffix}"
    silver_table = f"{catalog_name}.{silver_schema_name}.clean_{table_suffix}"

    print(f"Processing Bronze table: {bronze_table}")
    print(f"Writing Silver table: {silver_table}")

    # =========================
    # Read from Bronze
    # =========================

    bronze_df = spark.table(bronze_table)

    # =========================
    # Basic Silver Cleaning
    # =========================

    silver_df = (
        bronze_df
        # Standardize column names
        .transform(clean_column_names)

        # Add Silver processing metadata
        .withColumn("_silver_processed_timestamp", F.current_timestamp())
    )

    # =========================
    # Deduplication
    # =========================
    # Bronze already has _record_hash.
    # Keep the latest record for each _record_hash.

    window_spec = Window.partitionBy("_record_hash").orderBy(F.col("_ingest_timestamp").desc())

    silver_df = (
        silver_df
        .withColumn("_row_num", F.row_number().over(window_spec))
        .filter(F.col("_row_num") == 1)
        .drop("_row_num")
    )

    # =========================
    # Optional: Drop technical columns you don't want in Silver
    # =========================
    # Usually keep source lineage columns, but remove batch-level noise if not needed.

    columns_to_drop = [
        "_batch_id"
    ]

    existing_columns_to_drop = [c for c in columns_to_drop if c in silver_df.columns]

    silver_df = silver_df.drop(*existing_columns_to_drop)

    # =========================
    # Write to Silver Delta table
    # =========================

    (
        silver_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
    )

    print(f"Finished writing: {silver_table}")

Processing Bronze table: jarvis_training.bronze.raw_sqlserver_dbo_cards_data
Writing Silver table: jarvis_training.silver.clean_dbo_cards_data
Finished writing: jarvis_training.silver.clean_dbo_cards_data
Processing Bronze table: jarvis_training.bronze.raw_sqlserver_dbo_transactions_data
Writing Silver table: jarvis_training.silver.clean_dbo_transactions_data
Finished writing: jarvis_training.silver.clean_dbo_transactions_data
Processing Bronze table: jarvis_training.bronze.raw_sqlserver_dbo_users_data
Writing Silver table: jarvis_training.silver.clean_dbo_users_data
Finished writing: jarvis_training.silver.clean_dbo_users_data
